# Topic 13 - Async and Await

Practice the event loop, coroutines, tasks, and concurrent I/O. All code by hand. No Copilot.

## Learning objectives

- distinguish coroutine function, coroutine object, and Task
- use `await`, `asyncio.gather`, `asyncio.TaskGroup`
- identify blocking-the-loop bugs
- write concurrent async HTTP calls with `aiohttp`

## 1. Coroutine function vs coroutine vs Task

In [ ]:
# Jupyter already runs an event loop, so use top-level await (not asyncio.run()).
import asyncio
from typing import Coroutine

async def fetch_data(url: str) -> str:
    """Coroutine function."""
    await asyncio.sleep(0.1)  # simulate I/O
    return f"data from {url}"

# Calling the function returns a COROUTINE OBJECT (not running yet)
coro = fetch_data("http://example.com")
print(f"Type: {type(coro)}")
print(f"Is coroutine: {asyncio.iscoroutine(coro)}")

# Must be awaited or scheduled to run
result = await coro
print(f"Result: {result}")

## 2. Creating and running Tasks

In [ ]:
# Jupyter already runs an event loop, so use top-level await (not asyncio.run()).
async def main():
    # create_task schedules the coroutine on the event loop
    task1 = asyncio.create_task(fetch_data("http://api1.com"))
    task2 = asyncio.create_task(fetch_data("http://api2.com"))
    
    print(f"Task1 type: {type(task1)}")
    print(f"Task2 type: {type(task2)}")
    
    # Both are now running concurrently
    result1 = await task1
    result2 = await task2
    
    print(f"Result 1: {result1}")
    print(f"Result 2: {result2}")

await main()

## 3. `asyncio.gather` — concurrent execution

The idiomatic way to run multiple coroutines concurrently.

In [ ]:
# Jupyter already runs an event loop, so use top-level await (not asyncio.run()).
async def fetch_with_delay(url: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return f"{url} done after {delay}s"

async def main():
    import time
    start = time.perf_counter()
    
    # These run CONCURRENTLY - total time ≈ max delay (0.3s), not sum (0.6s)
    results = await asyncio.gather(
        fetch_with_delay("api1", 0.1),
        fetch_with_delay("api2", 0.2),
        fetch_with_delay("api3", 0.3),
    )
    
    elapsed = time.perf_counter() - start
    print(f"Elapsed: {elapsed:.2f}s")
    for r in results:
        print(f"  {r}")

await main()

## 4. `asyncio.TaskGroup` (Python 3.11+) — structured concurrency

In [ ]:
# Jupyter already runs an event loop, so use top-level await (not asyncio.run()).
async def main():
    async with asyncio.TaskGroup() as tg:
        task1 = tg.create_task(fetch_with_delay("api1", 0.1))
        task2 = tg.create_task(fetch_with_delay("api2", 0.2))
        task3 = tg.create_task(fetch_with_delay("api3", 0.3))
    # All tasks completed here (or exception raised)
    print(f"Task1: {task1.result()}")
    print(f"Task2: {task2.result()}")
    print(f"Task3: {task3.result()}")

await main()

## 5. Error handling with `gather`

`return_exceptions=True` prevents one failure from cancelling others.

In [ ]:
# Jupyter already runs an event loop, so use top-level await (not asyncio.run()).
async def might_fail(url: str, should_fail: bool) -> str:
    await asyncio.sleep(0.1)
    if should_fail:
        raise ValueError(f"Failed: {url}")
    return f"Success: {url}"

async def main():
    # Without return_exceptions: first exception propagates, others cancelled
    results = await asyncio.gather(
        might_fail("api1", False),
        might_fail("api2", True),
        might_fail("api3", False),
        return_exceptions=True  # Capture exceptions as results
    )
    
    for i, result in enumerate(results):
        if isinstance(result, Exception):
            print(f"Task {i+1}: FAILED - {result}")
        else:
            print(f"Task {i+1}: {result}")

await main()

## 6. Blocking the loop — the pitfall

Never use blocking I/O (`requests`, `time.sleep`, `psycopg2`) in async code.

In [ ]:
# Jupyter already runs an event loop, so use top-level await (not asyncio.run()).
import time

async def blocking_sleep():
    time.sleep(0.2)  # BLOCKS the entire event loop!
    return "done"

async def async_sleep():
    await asyncio.sleep(0.2)  # Yields control
    return "done"

async def main():
    import time
    
    # This takes ~0.6s (sequential because blocking)
    start = time.perf_counter()
    await asyncio.gather(blocking_sleep(), blocking_sleep(), blocking_sleep())
    print(f"Blocking: {time.perf_counter() - start:.2f}s")
    
    # This takes ~0.2s (concurrent)
    start = time.perf_counter()
    await asyncio.gather(async_sleep(), async_sleep(), async_sleep())
    print(f"Async: {time.perf_counter() - start:.2f}s")

await main()

## 7. Async HTTP with `aiohttp` (pattern for LLM calls)

In [ ]:
# This is the PATTERN you'll use for real LLM API calls
# (Requires `aiohttp` installed - dry run shows structure)

async def fetch_json(session, url: str) -> dict:
    async with session.get(url) as response:
        response.raise_for_status()
        return await response.json()

async def fetch_all(session, urls: list[str]) -> list[dict]:
    tasks = [fetch_json(session, url) for url in urls]
    return await asyncio.gather(*tasks, return_exceptions=True)

# Usage pattern:
# async with aiohttp.ClientSession() as session:
#     results = await fetch_all(session, urls)

print("Pattern defined. Install aiohttp to run live.")

## 8. Running sync code in thread pool (escape hatch)

In [ ]:
import requests

async def run_sync_request(url: str) -> dict:
    """Run blocking requests.get in thread pool."""
    loop = asyncio.get_event_loop()
    # run_in_executor runs the sync function in a thread
    response = await loop.run_in_executor(None, requests.get, url)
    return response.json()

# Only use when no async alternative exists
print("Escape hatch pattern defined.")

## 9. Async context manager pattern

In [ ]:
class AsyncClient:
    def __init__(self, base_url: str):
        self.base_url = base_url
        self.session = None
    
    async def __aenter__(self):
        import aiohttp
        self.session = aiohttp.ClientSession()
        return self
    
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        if self.session:
            await self.session.close()
    
    async def get(self, path: str) -> dict:
        url = f"{self.base_url}{path}"
        async with self.session.get(url) as resp:
            resp.raise_for_status()
            return await resp.json()

# Usage:
# async with AsyncClient("http://api.example.com") as client:
#     data = await client.get("/items")

print("Async context manager pattern defined.")

## 10. Type hints for async functions

In [ ]:
from typing import Awaitable, Coroutine

# A coroutine function
async def fetch(url: str) -> dict: ...

# Accepts any awaitable (coroutine, Task, Future)
async def process(awaitable: Awaitable[dict]) -> dict:
    return await awaitable

# Explicit coroutine type
async def wrapper() -> Coroutine[None, None, str]:
    return "hello"

print("Async type hints demonstrated.")

## 11. Completion exercises

Do these by hand. No autocomplete.

In [ ]:
# EXERCISE 1: Write an async function that fetches from 3 URLs
# with a semaphore limiting concurrency to 2

# EXERCISE 2: Create a retry wrapper with exponential backoff
# that works with async functions

# EXERCISE 3: Write a function that processes items in batches
# of N concurrently, waiting for each batch to complete

# EXERCISE 4: Identify the blocking call in this code:
# async def bad():
#     import json
#     with open('data.json') as f:  # <-- this
#         data = json.load(f)
#     return data

print("Exercises ready — implement by hand.")